[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/37_dpo_loss.ipynb)

# 🔴 Hard: DPO Loss

Реализуйте **Direct Preference Optimization (DPO)** loss. DPO обучает policy предпочитать выбранный ответ (`chosen`) отклонённому (`rejected`) на данных попарных предпочтений, используя reference policy как неподвижную точку отсчёта.

$$\mathcal{L}_{\text{DPO}} = -\log \sigma\Big(\beta \big[\log\frac{\pi(y_w)}{\text{ref}(y_w)} - \log\frac{\pi(y_l)}{\text{ref}(y_l)}\big]\Big)$$

## Разберите выражение на части

Сначала вычислите preference margin текущей policy: $m_\pi=\log\pi(y_w)-\log\pi(y_l)$. Аналогично reference margin: $m_\text{ref}=\log\pi_\text{ref}(y_w)-\log\pi_\text{ref}(y_l)$. Логит классификации предпочтения равен $z=\beta(m_\pi-m_\text{ref})$, а loss — `-log(sigmoid(z))`, усреднённый по batch.

Интуитивно DPO награждает не просто большой отрыв chosen от rejected, а **улучшение этого отрыва относительно reference model**. Если policy и reference дают одинаковые margins, `z=0` и loss одного примера равен $\log 2$. Чем больше положительный `z`, тем меньше loss. Параметр $\beta$ масштабирует чувствительность sigmoid и силу отклонения от reference.

### Числовой ориентир

Если policy margin равен `1.5`, reference margin `0.5`, а `beta=0.1`, то `z=0.1`. Не нужно переводить log-probabilities обратно в probabilities: разности логарифмов устойчивее и ровно соответствуют логарифму отношения.

### Signature
```python
def dpo_loss(policy_chosen_logps, policy_rejected_logps,
             ref_chosen_logps, ref_rejected_logps, beta=0.1) -> Tensor:
    # All inputs: (B,) log-probabilities
    # Returns: scalar loss
```

## План реализации

1. Получите две разности chosen-minus-rejected.
2. Вычтите reference margin из policy margin и умножьте на `beta`.
3. Превратите логит в отрицательный log-sigmoid и усредните. Для численной устойчивости полезны `F.logsigmoid` или эквивалент через softplus.

Reference log-probabilities считаются константами. В реальном pipeline их обычно заранее вычисляют или получают от frozen model. В этом упражнении градиенты нужны только по policy tensors.

## Быстрые проверки

- Одинаковые policy/reference margins дают loss около `log(2)`.
- Увеличение policy chosen log-probability должно уменьшать loss.
- Увеличение policy rejected log-probability должно увеличивать loss.
- Перестановка chosen и rejected меняет направление предпочтения.

## Частые ошибки

- Вычитать chosen policy из chosen reference отдельно, но неверно расставить знаки rejected.
- Применять sigmoid к каждому log-probability до построения margin.
- Возвращать вектор `(B,)` вместо scalar mean.
- Использовать `log(sigmoid(z))` напрямую при больших отрицательных `z`, получая численную нестабильность.

## Материалы для глубокого изучения

- [Direct Preference Optimization](https://arxiv.org/abs/2305.18290) — исходная статья и вывод objective.
- [Hugging Face TRL: DPO Trainer](https://huggingface.co/docs/trl/dpo_trainer) — практические варианты loss и параметры обучения.
- [PyTorch log-sigmoid](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.logsigmoid.html) — устойчивая реализация ключевой операции.

## Вопросы для самопроверки

1. Зачем DPO нужна reference policy?
2. Чему равен loss, когда оба preference margins совпадают?
3. Какие два входа должны получать противоположно направленные градиенты?

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn.functional as F

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def dpo_loss(policy_chosen_logps, policy_rejected_logps,
             ref_chosen_logps, ref_rejected_logps, beta=0.1):
    pass  # -log(sigmoid(beta * (chosen_reward - rejected_reward)))

In [ ]:
# 🧪 Debug
chosen = torch.tensor([0.0, 0.0])
rejected = torch.tensor([-5.0, -5.0])
ref_c = torch.tensor([-1.0, -1.0])
ref_r = torch.tensor([-1.0, -1.0])
print('Loss:', dpo_loss(chosen, rejected, ref_c, ref_r, beta=0.1).item())

In [ ]:
# ✅ SUBMIT
from torch_judge import check
check('dpo_loss')